# Notebook 01 — EDA + Preprocessing
### Paper: Profit-Driven Telecom Churn Prediction (IEEE CSDE 2026)

**Datasets:** Maven Telecom (primary) + Cell2Cell (independent generalization)

**Kaggle inputs to add:**
- `shilongzhuang/telecom-customer-churn-by-maven-analytics`
- `jpacse/datasets-for-churn-telecom`

**Output:** cleaned, stratified train/calibration/test splits saved to `/kaggle/working` (encoding + statistical imputation happen in Notebook 02, fit on train only).

**Split policy:** train / calibration / test = 60 / 20 / 20, stratified, `seed=42`
- **train** → fit base model
- **calibration** → fit probability calibrator + tune profit threshold (Notebook 02+)
- **test** → final untouched evaluation

In [1]:
import os, glob
import pandas as pd

# 1. Kaggle input e ki ki file ache dekho
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/jpacse/datasets-for-churn-telecom/cell2celltrain.csv
/kaggle/input/datasets/jpacse/datasets-for-churn-telecom/cell2cellholdout.csv
/kaggle/input/datasets/shilongzhuang/telecom-customer-churn-by-maven-analytics/telecom_customer_churn.csv
/kaggle/input/datasets/shilongzhuang/telecom-customer-churn-by-maven-analytics/telecom_zipcode_population.csv
/kaggle/input/datasets/shilongzhuang/telecom-customer-churn-by-maven-analytics/telecom_data_dictionary.csv


In [2]:
# 2. Maven load kore full columns dekho (path ta step 1 er output theke boshao)
maven = pd.read_csv('/kaggle/input/datasets/shilongzhuang/telecom-customer-churn-by-maven-analytics/telecom_customer_churn.csv')
print("MAVEN shape:", maven.shape)
print("MAVEN columns:\n", list(maven.columns))
# label column খুঁজে dekho
for c in maven.columns:
    if 'status' in c.lower() or 'churn' in c.lower():
        print(c, "->", maven[c].unique()[:5])

# 3. Cell2Cell load
cell = pd.read_csv('/kaggle/input/datasets/jpacse/datasets-for-churn-telecom/cell2celltrain.csv')
print("\nCELL2CELL shape:", cell.shape)
print("CELL2CELL columns:\n", list(cell.columns))
print("Churn dist:\n", cell['Churn'].value_counts())

MAVEN shape: (7043, 38)
MAVEN columns:
 ['Customer ID', 'Gender', 'Age', 'Married', 'Number of Dependents', 'City', 'Zip Code', 'Latitude', 'Longitude', 'Number of Referrals', 'Tenure in Months', 'Offer', 'Phone Service', 'Avg Monthly Long Distance Charges', 'Multiple Lines', 'Internet Service', 'Internet Type', 'Avg Monthly GB Download', 'Online Security', 'Online Backup', 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charge', 'Total Charges', 'Total Refunds', 'Total Extra Data Charges', 'Total Long Distance Charges', 'Total Revenue', 'Customer Status', 'Churn Category', 'Churn Reason']
Customer Status -> ['Stayed' 'Churned' 'Joined']
Churn Category -> [nan 'Competitor' 'Dissatisfaction' 'Other' 'Price']
Churn Reason -> [nan 'Competitor had better devices' 'Product dissatisfaction'
 'Network reliability' 'Limited range of services']

CELL2CELL shape:

In [3]:
# ===== CELL 1: imports + config =====
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
 
SEED = 42
np.random.seed(SEED)
 
OUT_DIR = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)
 
pd.set_option("display.max_columns", 100)

In [4]:
# ===== CELL 2: locate input files =====
# Run once to confirm exact paths, then hard-code below if they differ.
for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/jpacse/datasets-for-churn-telecom/cell2celltrain.csv
/kaggle/input/datasets/jpacse/datasets-for-churn-telecom/cell2cellholdout.csv
/kaggle/input/datasets/shilongzhuang/telecom-customer-churn-by-maven-analytics/telecom_customer_churn.csv
/kaggle/input/datasets/shilongzhuang/telecom-customer-churn-by-maven-analytics/telecom_zipcode_population.csv
/kaggle/input/datasets/shilongzhuang/telecom-customer-churn-by-maven-analytics/telecom_data_dictionary.csv


In [5]:
# ===== CELL 3: load raw datasets =====
MAVEN_PATH = "/kaggle/input/datasets/shilongzhuang/telecom-customer-churn-by-maven-analytics/telecom_customer_churn.csv"
CELL_PATH  = "/kaggle/input/datasets/jpacse/datasets-for-churn-telecom/cell2celltrain.csv"
 
maven_raw = pd.read_csv(MAVEN_PATH)
cell_raw  = pd.read_csv(CELL_PATH)
 
print("MAVEN:", maven_raw.shape)
print("CELL2CELL:", cell_raw.shape)

MAVEN: (7043, 38)
CELL2CELL: (51047, 58)


In [6]:
# ===== CELL 4: MAVEN cleaning =====
# ============================================================================
def clean_maven(df):
    df = df.copy()
 
    # --- target: drop "Joined" (new customers, no churn outcome), binary label ---
    df = df[df["Customer Status"].isin(["Stayed", "Churned"])].copy()
    df["target"] = (df["Customer Status"] == "Churned").astype(int)
 
    # --- keep churn reason aside for later SHAP validation (NOT a feature) ---
    churn_reason = df[["Customer ID", "Churn Category", "Churn Reason"]].copy()
 
    # --- drop leakage + id + high-cardinality geo ---
    leak_cols = ["Customer Status", "Churn Category", "Churn Reason"]
    id_geo    = ["Customer ID", "City", "Zip Code", "Latitude", "Longitude"]
    df = df.drop(columns=leak_cols + id_geo)
 
    # --- structural NaN fills (rule-based -> safe before split) ---
    # service-dependent categoricals become NaN when the parent service is absent
    service_cat = [
        "Multiple Lines", "Internet Type", "Online Security", "Online Backup",
        "Device Protection Plan", "Premium Tech Support", "Streaming TV",
        "Streaming Movies", "Streaming Music", "Unlimited Data",
    ]
    for c in service_cat:
        if c in df.columns:
            df[c] = df[c].fillna("None")
 
    # service-dependent numerics -> 0 when service absent (structural, not statistical)
    service_num = ["Avg Monthly Long Distance Charges", "Avg Monthly GB Download"]
    for c in service_num:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
 
    # 'Offer' -> "None" when no offer
    if "Offer" in df.columns:
        df["Offer"] = df["Offer"].fillna("None")
 
    # coerce remaining money columns to numeric (do NOT statistically impute here)
    money = ["Monthly Charge", "Total Charges", "Total Refunds",
             "Total Extra Data Charges", "Total Long Distance Charges", "Total Revenue"]
    for c in money:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
 
    return df, churn_reason
 
maven, maven_churn_reason = clean_maven(maven_raw)
print("MAVEN cleaned:", maven.shape)
print("Churn rate:", maven["target"].mean().round(4))
print("Remaining NaNs (top):")
print(maven.isna().sum()[maven.isna().sum() > 0])

MAVEN cleaned: (6589, 31)
Churn rate: 0.2837
Remaining NaNs (top):
Series([], dtype: int64)


In [7]:
# ===== CELL 5: CELL2CELL cleaning =====
# ============================================================================
def clean_cell2cell(df):
    df = df.copy()
 
    # --- target ---
    df["target"] = (df["Churn"] == "Yes").astype(int)
 
    # --- drop id, raw label, high-cardinality service area ---
    drop_cols = ["CustomerID", "Churn", "ServiceArea"]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
 
    # --- 'HandsetPrice' often has "Unknown" -> treat as NaN, keep numeric ---
    if "HandsetPrice" in df.columns:
        df["HandsetPrice"] = pd.to_numeric(df["HandsetPrice"], errors="coerce")
 
    # NOTE: numeric NaNs (MonthlyRevenue, MonthlyMinutes, AgeHH1/2, etc.) are left as-is.
    #       Median imputation is done in Notebook 02 pipeline (fit on train only).
    return df
 
cell = clean_cell2cell(cell_raw)
print("CELL2CELL cleaned:", cell.shape)
print("Churn rate:", cell["target"].mean().round(4))
print("Remaining NaNs (top):")
print(cell.isna().sum()[cell.isna().sum() > 0].head(20))

CELL2CELL cleaned: (51047, 56)
Churn rate: 0.2882
Remaining NaNs (top):
MonthlyRevenue             156
MonthlyMinutes             156
TotalRecurringCharge       156
DirectorAssistedCalls      156
OverageMinutes             156
RoamingCalls               156
PercChangeMinutes          367
PercChangeRevenues         367
Handsets                     1
HandsetModels                1
CurrentEquipmentDays         1
AgeHH1                     909
AgeHH2                     909
HandsetPrice             28982
dtype: int64


In [8]:
# ===== CELL 6: quick EDA =====
# ============================================================================
def quick_eda(df, name):
    print(f"\n========== {name} ==========")
    print("shape:", df.shape)
    print("churn rate:", df["target"].mean().round(4))
    num = df.select_dtypes(include=[np.number]).columns.drop("target", errors="ignore")
    cat = df.select_dtypes(exclude=[np.number]).columns
    print(f"numeric features: {len(num)} | categorical features: {len(cat)}")
    print("numeric summary:")
    print(df[num].describe().T[["mean", "std", "min", "max"]].round(2))
 
quick_eda(maven, "MAVEN")
quick_eda(cell, "CELL2CELL")
 
# optional plots (uncomment on Kaggle)
# import matplotlib.pyplot as plt
# maven["target"].value_counts().plot(kind="bar", title="Maven churn"); plt.show()
# cell["target"].value_counts().plot(kind="bar", title="Cell2Cell churn"); plt.show()


========== MAVEN ==========
shape: (6589, 31)
churn rate: 0.2837
numeric features: 12 | categorical features: 18
numeric summary:
                                      mean      std    min       max
Age                                  46.76    16.84  19.00     80.00
Number of Dependents                  0.48     0.97   0.00      9.00
Number of Referrals                   2.02     3.02   0.00     11.00
Tenure in Months                     34.50    23.97   1.00     72.00
Avg Monthly Long Distance Charges    23.00    15.47   0.00     49.99
Avg Monthly GB Download              20.88    20.41   0.00     85.00
Monthly Charge                       65.03    31.10 -10.00    118.75
Total Charges                      2432.04  2265.50  18.85   8684.80
Total Refunds                         2.08     8.13   0.00     49.79
Total Extra Data Charges              7.17    25.80   0.00    150.00
Total Long Distance Charges         798.09   853.77   0.00   3564.72
Total Revenue                      3235.2

In [9]:
# ===== CELL 7: stratified 60/20/20 split (train / calibration / test) =====
# ============================================================================
def split_60_20_20(df, name, seed=SEED):
    y = df["target"]
    # first: 60 train / 40 temp
    train_df, temp_df = train_test_split(
        df, test_size=0.40, stratify=y, random_state=seed
    )
    # then: split temp 50/50 -> 20 calibration / 20 test
    cal_df, test_df = train_test_split(
        temp_df, test_size=0.50, stratify=temp_df["target"], random_state=seed
    )
    for part, d in [("train", train_df), ("cal", cal_df), ("test", test_df)]:
        print(f"{name} {part}: {d.shape}, churn={d['target'].mean().round(4)}")
    return train_df, cal_df, test_df
 
print("\n--- MAVEN splits ---")
m_train, m_cal, m_test = split_60_20_20(maven, "MAVEN")
print("\n--- CELL2CELL splits ---")
c_train, c_cal, c_test = split_60_20_20(cell, "CELL2CELL")


--- MAVEN splits ---
MAVEN train: (3953, 31), churn=0.2836
MAVEN cal: (1318, 31), churn=0.2838
MAVEN test: (1318, 31), churn=0.2838

--- CELL2CELL splits ---
CELL2CELL train: (30628, 56), churn=0.2882
CELL2CELL cal: (10209, 56), churn=0.2882
CELL2CELL test: (10210, 56), churn=0.2881


In [10]:
# ===== CELL 8: save cleaned splits + metadata =====
# ============================================================================
m_train.to_csv(f"{OUT_DIR}/maven_train.csv", index=False)
m_cal.to_csv(f"{OUT_DIR}/maven_cal.csv", index=False)
m_test.to_csv(f"{OUT_DIR}/maven_test.csv", index=False)
 
c_train.to_csv(f"{OUT_DIR}/cell_train.csv", index=False)
c_cal.to_csv(f"{OUT_DIR}/cell_cal.csv", index=False)
c_test.to_csv(f"{OUT_DIR}/cell_test.csv", index=False)
 
# churn reason (Maven) for later SHAP validation
maven_churn_reason.to_csv(f"{OUT_DIR}/maven_churn_reason.csv", index=False)
 
# feature-type metadata (used by Notebook 02 ColumnTransformer)
def save_meta(df, name):
    num = list(df.select_dtypes(include=[np.number]).columns.drop("target", errors="ignore"))
    cat = list(df.select_dtypes(exclude=[np.number]).columns)
    meta = pd.DataFrame({"feature": num + cat,
                         "type": ["numeric"] * len(num) + ["categorical"] * len(cat)})
    meta.to_csv(f"{OUT_DIR}/{name}_feature_meta.csv", index=False)
    print(f"{name}: {len(num)} numeric, {len(cat)} categorical")
 
save_meta(maven, "maven")
save_meta(cell, "cell")
 
print("\nSaved files:")
for f in sorted(os.listdir(OUT_DIR)):
    print(" ", f)
 
# ============================================================================
# DONE. Next: Notebook 02 — baseline models (LR/RF/XGB/LGBM) with a leakage-safe
# ColumnTransformer (median-impute numeric, one-hot categoricals) fit on train.
# ============================================================================

maven: 12 numeric, 18 categorical
cell: 35 numeric, 20 categorical

Saved files:
  __notebook__.ipynb
  cell_cal.csv
  cell_feature_meta.csv
  cell_test.csv
  cell_train.csv
  maven_cal.csv
  maven_churn_reason.csv
  maven_feature_meta.csv
  maven_test.csv
  maven_train.csv
